In [7]:
tag2text_path="/workspaces/collaborative-scene-graphs/src/curb_projection/saved_models/tag2text_swin_14m.pth"
gd_config="/workspaces/collaborative-scene-graphs/src/curb_projection/saved_models/GroundingDINO_SwinT_OGC_config.py"
gd_weights="/workspaces/collaborative-scene-graphs/src/curb_projection/saved_models/groundingdino_swint_ogc.pth"
tap_path="/workspaces/collaborative-scene-graphs/src/curb_projection/saved_models/tap_vit_l_v1_1.pkl"
# tap_merge_path="/workspaces/collaborative-scene-graphs/src/OpenGraph/third_parties/tokenize-anything/weights/merged_2560.pkl"
# tap_concept_weights="/workspaces/collaborative-scene-graphs/src/OpenGraph/third_parties/tokenize-anything/weights/coco_80.pkl"
tap_concept_weights="/workspaces/collaborative-scene-graphs/src/curb_projection/saved_models/tap_concept_weights_cityscapes_19.pkl"

In [8]:
from ram.models import tag2text
delete_tag_index = list(range(3012, 3429))
# load model
tagging_model = tag2text(pretrained=tag2text_path,
                                        image_size=384,
                                        vit='swin_b',
                                        delete_tag_index=delete_tag_index)
tagging_model = tagging_model.eval().to("cuda")

/encoder/layer/0/crossattention/self/query is tied
/encoder/layer/0/crossattention/self/key is tied
/encoder/layer/0/crossattention/self/value is tied
/encoder/layer/0/crossattention/output/dense is tied
/encoder/layer/0/crossattention/output/LayerNorm is tied
/encoder/layer/0/intermediate/dense is tied
/encoder/layer/0/output/dense is tied
/encoder/layer/0/output/LayerNorm is tied
/encoder/layer/1/crossattention/self/query is tied
/encoder/layer/1/crossattention/self/key is tied
/encoder/layer/1/crossattention/self/value is tied
/encoder/layer/1/crossattention/output/dense is tied
/encoder/layer/1/crossattention/output/LayerNorm is tied
/encoder/layer/1/intermediate/dense is tied
/encoder/layer/1/output/dense is tied
/encoder/layer/1/output/LayerNorm is tied
--------------
/workspaces/collaborative-scene-graphs/src/curb_projection/saved_models/tag2text_swin_14m.pth
--------------
load checkpoint from /workspaces/collaborative-scene-graphs/src/curb_projection/saved_models/tag2text_swin

In [9]:
from groundingdino.util.inference import Model as GroundingDinoModel
grounding_dino_model = GroundingDinoModel(
    model_config_path = gd_config, 
    model_checkpoint_path = gd_weights, 
    device="cuda"
)

final text_encoder_type: bert-base-uncased


In [10]:

# TAP
from tokenize_anything import model_registry
tap_model = model_registry["tap_vit_l"](checkpoint=tap_path)
tap_model.concept_projector.reset_weights(tap_concept_weights)
tap_model.text_decoder.reset_cache(max_batch_size=1000)

# mask_generator = MyAutomaticMaskGenerator(tagging_model=tagging_model, grounding_dino_model=grounding_dino_model, tap_model=tap_model)


In [11]:
import amg_class
from importlib import reload
amg_class = reload(amg_class)
mask_generator = amg_class.MyAutomaticMaskGenerator(tagging_model=tagging_model, grounding_dino_model=grounding_dino_model, tap_model=tap_model)

In [12]:
import os
import cv2
from pathlib import Path
from tqdm import tqdm

testset = "/workspaces/collaborative-scene-graphs/src/curb_testset_creator/test/"
fnames = os.listdir(testset)
fnames.sort()

for i in tqdm(range(5)):
    # for i in tqdm(range(len(fnames))):
    img = cv2.imread(testset + fnames[i])
    res = mask_generator.generate(
        img, save_path=Path(fnames[i]), save_vis=True
    )

  0%|          | 0/5 [00:00<?, ?it/s]

tag2text Model output time: 0.21696686744689941 seconds
dino Model output time: 0.16085481643676758 seconds
tap Model output time: 0.10584378242492676 seconds
tap concepts: ['car', 'car', 'traffic light', 'traffic light', 'person', 'person', 'car', 'person', 'person', 'person', 'car', 'building', 'person', 'road', 'car', 'building', 'fence', 'traffic light', 'person', 'building', 'pole', 'pole', 'person', 'wall', 'person', 'traffic light', 'sidewalk', 'pole', 'person', 'pole']


 20%|██        | 1/5 [00:06<00:26,  6.52s/it]

tag2text Model output time: 0.18997597694396973 seconds
dino Model output time: 0.15856409072875977 seconds
tap Model output time: 0.10313558578491211 seconds
tap concepts: ['car', 'terrain', 'person', 'person', 'building', 'sidewalk', 'vegetation', 'pole', 'road', 'car', 'person', 'pole', 'person', 'person', 'person', 'person', 'person', 'sidewalk', 'person', 'building', 'bicycle', 'person', 'wall', 'car', 'person', 'building', 'bicycle', 'person', 'person', 'person', 'bicycle']


 40%|████      | 2/5 [00:13<00:19,  6.63s/it]

...

 60%|██████    | 3/5 [00:19<00:13,  6.54s/it]

tag2text Model output time: 0.18282413482666016 seconds
dino Model output time: 0.1598217487335205 seconds
tap Model output time: 0.10817265510559082 seconds
tap concepts: ['person', 'person', 'person', 'person', 'car', 'car', 'sidewalk', 'car', 'car', 'pole', 'person', 'road', 'car', 'building', 'car', 'car', 'car', 'traffic sign', 'building', 'pole', 'car', 'bicycle', 'pole', 'pole', 'person', 'person', 'fence', 'wall', 'person', 'car', 'person', 'person', 'bicycle', 'bicycle', 'traffic sign', 'person', 'person']


 80%|████████  | 4/5 [00:27<00:07,  7.07s/it]

tag2text Model output time: 0.24097776412963867 seconds
dino Model output time: 0.1709294319152832 seconds
tap Model output time: 0.10903739929199219 seconds
tap concepts: ['car', 'person', 'car', 'person', 'person', 'traffic light', 'pole', 'person', 'sidewalk', 'person', 'road', 'wall', 'pole', 'pole', 'person', 'building', 'person', 'person', 'pole', 'car', 'vegetation', 'person', 'person', 'car', 'wall', 'wall', 'car', 'person', 'person', 'building']


100%|██████████| 5/5 [00:34<00:00,  6.85s/it]
